# ViT Positional Embedding Comparison

This notebook trains two variants of the Vision Transformer (ViT) on CIFAR-10 from scratch:
1. **Original ViT**: with a 1D learned additive positional embedding.
2. **Modified ViT**: with a 2D Rotary Position Embedding (RoPE).

This is for the assignment described in `instruction/01_problem_statement.md`.

**GitHub Repository:** [Sagnik120/vit-positional-embedding-comparison](https://github.com/Sagnik120/vit-positional-embedding-comparison)

> **IMPORTANT:** Before running this notebook, go to **Runtime > Change runtime type** and select **T4 GPU**.

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU Name: Tesla T4


In [4]:

# Navigate to your Google Drive
%cd /content/drive/MyDrive

# Clean up any old copies if they exist, then clone
!rm -rf vit-positional-embedding-comparison
!git clone https://github.com/Sagnik120/vit-positional-embedding-comparison.git

# Move into the folder inside your Drive
%cd vit-positional-embedding-comparison



/content/drive/MyDrive
Cloning into 'vit-positional-embedding-comparison'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 77 (delta 11), reused 73 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 3.14 MiB | 13.81 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/drive/MyDrive/vit-positional-embedding-comparison


In [5]:
!pip install -q -r requirements.txt

In [6]:
!python scripts/download_data.py

[info] downloading CIFAR-10 train split to /content/drive/MyDrive/vit-positional-embedding-comparison/data/cifar10 ...
100% 170M/170M [29:01<00:00, 97.9kB/s]
[info] train set size: 50000
[info] downloading CIFAR-10 test split to /content/drive/MyDrive/vit-positional-embedding-comparison/data/cifar10 ...
[info] test set size: 10000
[done] CIFAR-10 ready.


In [7]:
!python tests/test_pipeline.py


--- 1. Environment / imports ---
torch=2.11.0+cu128, torchvision=0.26.0+cu128, einops=0.8.2
[PASS] 1. Environment / imports

--- 2. Device availability ---
selected device: cuda
[PASS] 2. Device availability

--- 3. Config sanity ---
image_size=32, patch_size=4, grid=8x8, num_patches=64, dim=256, heads=4, dim_head=64
[PASS] 3. Config sanity

--- 4. Dataset loader ---
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our 

In [8]:
!bash scripts/train_baseline.sh

[info] Starting Baseline ViT training (100 epochs)... please wait, this takes ~1 hour and prints at the end of each epoch.
[info] using device: cuda
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
[info] model=original params=3,191,146
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoad

In [9]:
!bash scripts/train_modified.sh

[info] Starting Modified ViT (RoPE) training (100 epochs)... please wait, this takes ~1 hour and prints at the end of each epoch.
[info] using device: cuda
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
[info] model=modified params=3,174,506
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get D

In [10]:
%cd src
!python -m common.evaluate --model original --out ../results/baseline
!python -m common.evaluate --model modified --out ../results/modified_rope
%cd ..

/content/drive/MyDrive/vit-positional-embedding-comparison/src
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
[info] loaded checkpoint from epoch 80 (best_val_acc=0.8428)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid

In [ ]:
!python scripts/evaluate_all.py

In [12]:
from IPython.display import Image, Markdown, display
import os

try:
    with open('results/comparison/comparison_table.md', 'r') as f:
        display(Markdown(f.read()))
except FileNotFoundError:
    print("Comparison table not found. Make sure evaluation completed successfully.")

if os.path.exists('results/comparison/combined_loss_curves.png'):
    display(Image(filename='results/comparison/combined_loss_curves.png'))
else:
    print("Loss curves not found.")

if os.path.exists('results/comparison/combined_accuracy_curves.png'):
    display(Image(filename='results/comparison/combined_accuracy_curves.png'))
else:
    print("Accuracy curves not found.")

# Comparison Table (auto-generated by scripts/evaluate_all.py)
(Top-1 test accuracy, best val accuracy, epoch of best val, generalization gap,
 for original vs. modified — filled in after both training runs complete.)


In [13]:
!zip -r vit_pe_comparison_results.zip results/ report/ docs/ CHANGES.md

  adding: results/ (stored 0%)
  adding: results/baseline/ (stored 0%)
  adding: results/baseline/checkpoints/ (stored 0%)
  adding: results/baseline/checkpoints/.gitkeep (stored 0%)
  adding: results/baseline/checkpoints/best.pt (deflated 9%)
  adding: results/baseline/logs/ (stored 0%)
  adding: results/baseline/logs/train_log.csv (deflated 53%)
  adding: results/baseline/logs/val_log.csv (stored 0%)
  adding: results/baseline/metrics/ (stored 0%)
  adding: results/baseline/metrics/best_val_checkpoint_info.json (deflated 24%)
  adding: results/baseline/metrics/test_accuracy.json (deflated 32%)
  adding: results/baseline/visualizations/ (stored 0%)
  adding: results/baseline/visualizations/accuracy_curves/ (stored 0%)
  adding: results/baseline/visualizations/accuracy_curves/.gitkeep (stored 0%)
  adding: results/baseline/visualizations/attention_maps/ (stored 0%)
  adding: results/baseline/visualizations/attention_maps/.gitkeep (stored 0%)
  adding: results/baseline/visualizations/lo

In [14]:
from google.colab import files
if os.path.exists('vit_pe_comparison_results.zip'):
    files.download('vit_pe_comparison_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
from google.colab import files
import os
# Optional: Download individual checkpoints separately in case the zip is too large
if os.path.exists('results/baseline/checkpoints/best.pt'):
    files.download('results/baseline/checkpoints/best.pt')
if os.path.exists('results/modified_rope/checkpoints/best.pt'):
    files.download('results/modified_rope/checkpoints/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>